# سلول ۱ — نصب پکیج‌ها


In [1]:
!pip install -q fastapi "uvicorn[standard]" nest-asyncio

# سلول ۲ — Setup: Drive، Device، Logging، پوشه‌ها


In [2]:
import os, sys, time, math, json, pickle, random, copy, re, socket, subprocess, asyncio, threading, unicodedata, logging
from pathlib import Path
from dataclasses import dataclass, asdict
from functools import lru_cache

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_ROOT = Path("/content/drive/MyDrive/persian_text_pipeline") if IN_COLAB else Path("./persian_text_pipeline")
DATA_DIR   = PROJECT_ROOT / "data"
CKPT_DIR   = PROJECT_ROOT / "checkpoints"
LOG_DIR    = PROJECT_ROOT / "logs"
TEST_DIR   = PROJECT_ROOT / "tests"

for d in [DATA_DIR, CKPT_DIR, LOG_DIR, TEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "pipeline.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger("persian_text_pipeline")
print(f"Project root: {PROJECT_ROOT}")

Mounted at /content/drive
Device: cuda
GPU: Tesla T4
Memory: 15.6 GB
Project root: /content/drive/MyDrive/persian_text_pipeline


# سلول ۳ — Config با @dataclass


In [3]:
@dataclass
class ModelConfig:
    src_vocab: int = 140
    tgt_vocab: int = 140
    d_model: int = 256
    N: int = 3
    heads: int = 8
    d_ff: int = 512
    dropout: float = 0.1
    max_len: int = 512
    def to_dict(self): return asdict(self)

@dataclass
class TrainConfig:
    epochs: int = 15
    batch_size: int = 64
    lr: float = 1e-3
    warmup_steps: int = 1000
    grad_clip: float = 1.0
    label_smoothing: float = 0.05
    grad_accum_steps: int = 2
    patience: int = 5
    max_seq_len: int = 64
    def to_dict(self): return asdict(self)

model_cfg = ModelConfig()
train_cfg = TrainConfig()
print("Model:", model_cfg.to_dict())
print("Train:", train_cfg.to_dict())

Model: {'src_vocab': 140, 'tgt_vocab': 140, 'd_model': 256, 'N': 3, 'heads': 8, 'd_ff': 512, 'dropout': 0.1, 'max_len': 512}
Train: {'epochs': 15, 'batch_size': 64, 'lr': 0.001, 'warmup_steps': 1000, 'grad_clip': 1.0, 'label_smoothing': 0.05, 'grad_accum_steps': 2, 'patience': 5, 'max_seq_len': 64}


# سلول ۴ — Sanitizer: constants


In [4]:
PERSIAN_LETTERS = "ابپتثجچحخدذرزژسشصضطظعغفقکگلمنوهی"

ARABIC_TO_PERSIAN = {"ك": "ک", "ي": "ی", "ة": "ه", "أ": "ا", "إ": "ا", "ؤ": "و", "ئ": "ی"}
DIACRITICS = set("ًٌٍَُِّْـ")

ARABIC_DIGITS  = "٠١٢٣٤٥٦٧٨٩"
PERSIAN_DIGITS = "۰۱۲۳۴۵۶۷۸۹"
ENGLISH_DIGITS = "0123456789"
DIGIT_MAP = {}
for i, ch in enumerate(ARABIC_DIGITS):  DIGIT_MAP[ch] = ENGLISH_DIGITS[i]
for i, ch in enumerate(PERSIAN_DIGITS): DIGIT_MAP[ch] = ENGLISH_DIGITS[i]

HTML_TAG_RE      = re.compile(r"<[^>]+>")
URL_RE           = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
EMAIL_RE         = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
MULTISPACE_RE    = re.compile(r"[ \t]+")
MULTINEWLINE_RE  = re.compile(r"\n{3,}")
REPEATED_CHAR_RE = re.compile(r"(.)\1{3,}")
CONTROL_CHARS_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")
print("✅ Sanitizer constants loaded")

✅ Sanitizer constants loaded


# سلول ۵ — Sanitizer: توابع تک‌وظیفه


In [5]:
def _nfkc(t):           return unicodedata.normalize("NFKC", t)
def _strip_controls(t): return CONTROL_CHARS_RE.sub("", t)
def _strip_html(t):     return HTML_TAG_RE.sub(" ", t)
def _strip_urls_emails(t): return EMAIL_RE.sub(" ", URL_RE.sub(" ", t))
def _ar2fa(t):          return t.translate(str.maketrans(ARABIC_TO_PERSIAN))
def _remove_diacritics(t): return "".join(ch for ch in t if ch not in DIACRITICS)
def _normalize_digits(t):  return t.translate(str.maketrans(DIGIT_MAP))

def _fix_zwnj(t):
    t = re.sub(r"\bمی\s+([آ-ی])", r"می‌\1", t)
    t = re.sub(r"\bنمی\s+([آ-ی])", r"نمی‌\1", t)
    t = re.sub(r"([آ-ی])\s+ها\b", r"\1‌ها", t)
    t = re.sub(r"([آ-ی])\s+های\b", r"\1‌های", t)
    t = re.sub(r"([آ-ی])\s+تر\b", r"\1‌تر", t)
    t = re.sub(r"([آ-ی])\s+ترین\b", r"\1‌ترین", t)
    return t

def _collapse_repeats(t):   return REPEATED_CHAR_RE.sub(r"\1", t)
def _collapse_whitespace(t): return MULTINEWLINE_RE.sub("\n\n", MULTISPACE_RE.sub(" ", t)).strip()
print("✅ Sanitizer functions loaded")

✅ Sanitizer functions loaded


# سلول ۶ — Sanitizer: pipeline


In [6]:
@dataclass
class SanitizeOptions:
    apply_zwnj: bool = True
    apply_digits: bool = True
    strip_urls: bool = True
    strip_html: bool = True
    collapse_repeats: bool = True
    remove_diacritics: bool = True


def sanitize(text: str, opts: SanitizeOptions = None) -> dict:
    opts = opts or SanitizeOptions()
    original = text
    steps = []
    def _run(name, fn):
        nonlocal text
        before = text
        text = fn(text)
        if text != before: steps.append(name)
    _run("nfkc", _nfkc)
    _run("controls", _strip_controls)
    if opts.strip_html:         _run("html", _strip_html)
    if opts.strip_urls:         _run("urls_emails", _strip_urls_emails)
    _run("ar2fa", _ar2fa)
    if opts.remove_diacritics:  _run("diacritics", _remove_diacritics)
    if opts.apply_digits:       _run("digits", _normalize_digits)
    if opts.apply_zwnj:         _run("zwnj", _fix_zwnj)
    if opts.collapse_repeats:   _run("repeats", _collapse_repeats)
    _run("whitespace", _collapse_whitespace)
    return {"text": text, "changed": text != original, "steps_applied": steps}


# تست سریع
print(sanitize("سلام   می روم كتَب هاي جديد <b>خوب</b> ٣ عدد ۱۲۳۴"))

{'text': 'سلام می\u200cروم کتب\u200cهای جدید خوب 3 عدد 1234', 'changed': True, 'steps_applied': ['html', 'ar2fa', 'diacritics', 'digits', 'zwnj', 'whitespace']}


# سلول ۷ — Vocab


In [7]:
FA_CHARS = list(
    PERSIAN_LETTERS + "ئءأإؤ"
    + ARABIC_DIGITS + ENGLISH_DIGITS
    + "abcdefghijklmnopqrstuvwxyz" + "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    + " .,;:!?؟،«»()-'\""
    + "\u200c" + "ًٌٍَُِّْ" + "ـ"
)
SPECIAL_TOKENS = ["<pad>", "<sos>", "<eos>", "<unk>"]
N_SPECIAL = len(SPECIAL_TOKENS)


class CharVocab:
    def __init__(self, chars):
        unique = []
        for c in chars:
            if c not in unique: unique.append(c)
        self.c2i = {c: i + N_SPECIAL for i, c in enumerate(unique)}
        self.i2c = {i: c for c, i in self.c2i.items()}
        self.pad_idx, self.sos_idx, self.eos_idx, self.unk_idx = 0, 1, 2, 3
        self.size = N_SPECIAL + len(unique)
    def encode(self, text, add_sos=False, add_eos=False):
        ids = [self.c2i.get(c, self.unk_idx) for c in text]
        if add_sos: ids = [self.sos_idx] + ids
        if add_eos: ids = ids + [self.eos_idx]
        return ids
    def decode(self, ids, skip_special=True):
        return "".join(self.i2c[int(i)] for i in ids if not (skip_special and int(i) < N_SPECIAL))


vocab = CharVocab(FA_CHARS)
model_cfg.src_vocab = model_cfg.tgt_vocab = vocab.size
print(f"Vocab size: {vocab.size}")

Vocab size: 139


# سلول ۸ — Model blocks (⭐ deepcopy فیکس اصلی)


In [8]:
def clones(module, N):
    """⭐ deepcopy — نه shared reference. ظرفیت N برابر می‌شود."""
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])


def subsequent_mask(size):
    return ~torch.triu(torch.ones(1, size, size, dtype=torch.bool), diagonal=1)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        assert d_model % heads == 0
        self.d_k = d_model // heads
        self.heads = heads
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.dropout = nn.Dropout(dropout)
    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        q, k, v = [lin(x).view(B, -1, self.heads, self.d_k).transpose(1, 2)
                   for lin, x in zip(self.linears, (q, k, v))]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            while mask.dim() < scores.dim(): mask = mask.unsqueeze(1)
            scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        out = torch.matmul(self.dropout(F.softmax(scores, dim=-1)), v)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.heads * self.d_k)
        return self.linears[3](out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x): return self.w2(self.dropout(F.relu(self.w1(x))))


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return self.dropout(x + self.pe[:, :x.size(1)])


class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.dropout1, self.dropout2 = nn.Dropout(dropout), nn.Dropout(dropout)
    def forward(self, x, mask):
        x = x + self.dropout1(self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x), mask))
        x = x + self.dropout2(self.feed_forward(self.norm2(x)))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model); self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout); self.dropout2 = nn.Dropout(dropout); self.dropout3 = nn.Dropout(dropout)
    def forward(self, x, memory, src_mask, tgt_mask):
        x = x + self.dropout1(self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x), tgt_mask))
        x = x + self.dropout2(self.cross_attn(self.norm2(x), memory, memory, src_mask))
        x = x + self.dropout3(self.feed_forward(self.norm3(x)))
        return x
print("✅ Model blocks loaded")

✅ Model blocks loaded


# سلول ۹ — Full Transformer


In [9]:
class Encoder(nn.Module):
    def __init__(self, layer, N, d_model):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x, mask):
        for l in self.layers: x = l(x, mask)
        return self.norm(x)


class Decoder(nn.Module):
    def __init__(self, layer, N, d_model):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x, memory, src_mask, tgt_mask):
        for l in self.layers: x = l(x, memory, src_mask, tgt_mask)
        return self.norm(x)


class Generator(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab)
    def forward(self, x): return F.log_softmax(self.proj(x), dim=-1)


class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model, N, heads, d_ff, dropout, max_len):
        super().__init__()
        self.encoder    = Encoder(EncoderLayer(d_model, heads, d_ff, dropout), N, d_model)
        self.decoder    = Decoder(DecoderLayer(d_model, heads, d_ff, dropout), N, d_model)
        self.src_embed  = nn.Embedding(src_vocab, d_model)
        self.tgt_embed  = nn.Embedding(tgt_vocab, d_model)
        self.pos_enc    = PositionalEncoding(d_model, max_len, dropout)
        self.generator  = Generator(d_model, tgt_vocab)
        self.d_model    = d_model
    def encode(self, src, src_mask):
        return self.encoder(self.pos_enc(self.src_embed(src) * math.sqrt(self.d_model)), src_mask)
    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.pos_enc(self.tgt_embed(tgt) * math.sqrt(self.d_model)), memory, src_mask, tgt_mask)
    def forward(self, src, tgt, src_mask, tgt_mask):
        return self.generator(self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask))
print("✅ Transformer loaded")

✅ Transformer loaded


# سلول ۱۰ — داده آموزش (۱۲۰ جمله + sanitize هر دو طرف)


In [10]:
random.seed(42)

SEED_SENTENCES = [
    "سلام حال شما چطور است", "امروز هوا خیلی خوب است", "من کتاب می‌خوانم",
    "او به مدرسه می‌رود", "ما در تهران زندگی می‌کنیم", "این غذا خیلی خوشمزه است",
    "کتاب روی میز است", "برنامه‌نویسی خیلی جذاب است", "دانشجو در کتابخانه مطالعه می‌کند",
    "هواپیما ساعت هشت پرواز می‌کند", "پدرم در شرکت کار می‌کند", "مادرم غذا می‌پزد",
    "بچه‌ها در حیاط بازی می‌کنند", "من فارسی یاد می‌گیرم", "کامپیوتر جدید خریدم",
    "امروز جلسه مهمی داریم", "این فیلم خیلی قشنگ بود", "شهر تهران بزرگ است",
    "درخت‌های باغ شکوفه داده‌اند", "موسیقی ایرانی زیباست", "چای داغ می‌نوشیم",
    "نتیجه امتحان فردا اعلام می‌شود", "من به موسیقی گوش می‌دهم", "هوای زمستان سرد است",
    "برف روی کوه‌ها نشسته است", "دوست من در دانشگاه درس می‌خواند", "این ساختمان بلند است",
    "ساعت ده شب به خانه رسیدم", "رنگ آسمان آبی است", "ماهی در دریا شنا می‌کند",
    "پرنده روی شاخه آواز می‌خواند", "غذا با برنج و خورش است", "او کتاب جدیدی نوشت",
    "قطار ساعت شش حرکت می‌کند", "درخت سیب در باغ است", "نقاشی روی دیوار آویزان است",
    "من هر روز ورزش می‌کنم", "همسایه ما مهندس است", "این ماشین خیلی سریع است",
    "بچه گربه در حیاط است", "من عاشق کتاب خواندن هستم", "او هر روز صبح زود بیدار می‌شود",
    "ما با هم به سینما رفتیم", "این گل بسیار زیبا است", "بچه‌ها در پارک فوتبال بازی می‌کنند",
    "استاد درس را با دقت توضیح می‌دهد", "دانشجویان برای امتحان آماده می‌شوند",
    "مادر برای بچه‌ها قصه می‌گوید", "پدر ماشین را تعمیر می‌کند",
    "خواهرم در بیمارستان کار می‌کند", "برادرم دانشجوی پزشکی است",
    "ساعت هشت صبح به اداره می‌روم", "ناهار را در رستوران خوردیم",
    "شام آماده است بیا بخوریم", "پنجره اتاق را باز کردم",
    "هوا بسیار دلپذیر است", "امروز روز خوبی بود", "فردا به مسافرت می‌رویم",
    "تابستان امسال خیلی گرم بود", "پاییز فصل زیبایی است", "زمستان با برف می‌آید",
    "بهار فصل شکوفه‌ها است", "من در شهر شیراز متولد شدم", "او اهل اصفهان است",
    "ما به شمال سفر کردیم", "دریا آرام و زیبا بود", "کوه‌های البرز بسیار بلند هستند",
    "جنگل‌های شمال دیدنی هستند", "کویر لوت شگفت‌انگیز است",
    "خلیج فارس زیبایی خاصی دارد", "این فرش ایرانی است",
    "صنایع دستی ایران معروف است", "زعفران ایرانی بهترین است",
    "پسته رفسنجان مرغوب است", "قالی کاشان هنر ایرانی است",
    "شعر حافظ دلنشین است", "مولانا شاعر بزرگی بود",
    "سعدی در شیراز زندگی می‌کرد", "فردوسی شاهنامه را سرود",
    "خیام رباعیات زیبایی دارد", "این کتاب درباره تاریخ است",
    "ریاضیات علم جالبی است", "فیزیک قوانین طبیعت را بررسی می‌کند",
    "شیمی درباره مواد است", "زیست شناسی علم حیات است",
    "کامپیوتر ابزار مفیدی است", "اینترنت دنیا را تغییر داد",
    "هوش مصنوعی آینده را می‌سازد", "موبایل جدید خریدم",
    "این لپ تاپ خیلی سریع است", "اینترنت امروز قطع شده است",
    "برق خانه قطع شده بود", "آب لوله‌کشی قطع شده است",
    "باران شدیدی بارید", "باد درختان را تکان می‌دهد",
    "ابرها در آسمان حرکت می‌کنند", "خورشید در حال غروب است",
    "ماه امشب کامل است", "ستارگان در شب می‌درخشند",
    "این آهنگ زیبا است", "موسیقی سنتی ایرانی دلنشین است",
    "این فیلم را دیدم", "سریال جدیدی شروع کردم",
    "کتاب جدیدی می‌خوانم", "داستان جالبی تعریف کرد",
    "این خبر مهمی است", "روزنامه امروز را خواندم",
    "این مجله ماهانه منتشر می‌شود", "به من ایمیل فرستاد",
    "پیامک دریافت کردم", "تلفن را جواب داد",
    "با دوستانم تماس گرفتم", "برای شما آرزوی موفقیت دارم",
    "تشکر می‌کنم از لطف شما", "خواهش می‌کنم بفرمایید",
    "ببخشید مزاحم شدم", "سلام دوباره صبح بخیر",
    "شب بخیر خواب خوبی داشته باشی", "روز خوبی داشته باشید",
    "پایان این جلسه است",
]
print(f"Seed sentences: {len(SEED_SENTENCES)}")

FA_LETTERS = list(PERSIAN_LETTERS)


def corrupt_word(word, rng):
    if len(word) <= 2: return word
    op = rng.choice(["drop", "insert", "substitute", "swap", "multi"])
    chars = list(word)
    if op == "drop":
        chars.pop(rng.randrange(len(chars)))
    elif op == "insert":
        chars.insert(rng.randrange(len(chars) + 1), rng.choice(FA_LETTERS))
    elif op == "substitute":
        chars[rng.randrange(len(chars))] = rng.choice(FA_LETTERS)
    elif op == "swap":
        i = rng.randrange(len(chars) - 1)
        chars[i], chars[i+1] = chars[i+1], chars[i]
    elif op == "multi":
        for _ in range(rng.randint(1, 2)):
            if len(chars) > 2 and rng.random() < 0.5:
                chars.pop(rng.randrange(len(chars)))
            else:
                chars[rng.randrange(len(chars))] = rng.choice(FA_LETTERS)
    return "".join(chars)


def corrupt_sentence(sentence, rng, p_word=0.35):
    return " ".join(corrupt_word(w, rng) if rng.random() < p_word else w
                    for w in sentence.split(" "))


def build_dataset(n_samples, rng):
    """⭐ sanitize هر دو طرف → توزیع آموزش == توزیع inference."""
    pairs = []
    for _ in range(n_samples):
        base = rng.choice(SEED_SENTENCES)
        if rng.random() < 0.3:
            base = (base + " " + rng.choice(SEED_SENTENCES)).strip()
        bad_raw = corrupt_sentence(base, rng, p_word=rng.uniform(0.25, 0.5))
        if bad_raw == base: continue
        bad_san  = sanitize(bad_raw)["text"]
        good_san = sanitize(base)["text"]
        if bad_san != good_san and bad_san and good_san:
            pairs.append((bad_san, good_san))
    return pairs


TRAIN_PATH = DATA_DIR / "train_pairs.pkl"
VAL_PATH   = DATA_DIR / "val_pairs.pkl"

if TRAIN_PATH.exists() and VAL_PATH.exists():
    print("Loading cached dataset...")
    with open(TRAIN_PATH, "rb") as f: train_pairs = pickle.load(f)
    with open(VAL_PATH, "rb") as f:   val_pairs   = pickle.load(f)
else:
    print("Generating dataset...")
    rng = random.Random(42)
    train_pairs = build_dataset(60_000, rng)
    val_pairs   = build_dataset(3_000, rng)
    with open(TRAIN_PATH, "wb") as f: pickle.dump(train_pairs, f)
    with open(VAL_PATH, "wb") as f:   pickle.dump(val_pairs, f)

print(f"Train: {len(train_pairs):,} | Val: {len(val_pairs):,}")
for i in range(3):
    print(f"\n[{i}] bad : {train_pairs[i][0]}")
    print(f"    good: {train_pairs[i][1]}")

Seed sentences: 120
Generating dataset...
Train: 51,558 | Val: 2,593

[0] bad : ریاضیت علم جالبی است اد دلختان را تان می‌دهد
    good: ریاضیات علم جالبی است باد درختان را تکان می‌دهد

[1] bad : ان گل بسیار زیبا است ساعت ده شب به خانه رسیدم
    good: این گل بسیار زیبا است ساعت ده شب به خانه رسیدم

[2] bad : یام رباعیات زیبایی دارد
    good: خیام رباعیات زیبایی دارد


# سلول ۱۱ — Dataset + Collate


In [11]:
class SpellDataset(Dataset):
    def __init__(self, pairs, vocab, max_len=64):
        self.samples = []
        for bad, good in pairs:
            src = vocab.encode(bad[:max_len])
            tgt = vocab.encode(good[:max_len], add_sos=True, add_eos=True)
            if src and tgt: self.samples.append((src, tgt))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]


def make_collate_fn(vocab, max_len):
    causal_full = subsequent_mask(max_len + 2)
    def collate_fn(batch):
        srcs, tgts = zip(*batch)
        max_src, max_tgt = max(len(s) for s in srcs), max(len(t) for t in tgts)
        B = len(batch)
        src_pad = torch.full((B, max_src), vocab.pad_idx, dtype=torch.long)
        tgt_pad = torch.full((B, max_tgt), vocab.pad_idx, dtype=torch.long)
        src_mask = torch.zeros(B, max_src, dtype=torch.bool)
        tgt_mask = torch.zeros(B, max_tgt, dtype=torch.bool)
        for i, (s, t) in enumerate(zip(srcs, tgts)):
            src_pad[i, :len(s)] = torch.tensor(s)
            tgt_pad[i, :len(t)] = torch.tensor(t)
            src_mask[i, :len(s)] = True
            tgt_mask[i, :len(t)] = True
        causal = causal_full[:, :max_tgt, :max_tgt]
        tgt_mask = (causal & tgt_mask.unsqueeze(1)) | torch.eye(max_tgt, dtype=torch.bool).unsqueeze(0)
        return {"src": src_pad, "tgt": tgt_pad,
                "src_mask": src_mask.unsqueeze(1), "tgt_mask": tgt_mask}
    return collate_fn


train_ds = SpellDataset(train_pairs, vocab, train_cfg.max_seq_len)
val_ds   = SpellDataset(val_pairs,   vocab, train_cfg.max_seq_len)
collate  = make_collate_fn(vocab, train_cfg.max_seq_len)

train_loader = DataLoader(train_ds, batch_size=train_cfg.batch_size, shuffle=True,
                          collate_fn=collate, drop_last=True, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=train_cfg.batch_size, shuffle=False,
                          collate_fn=collate, pin_memory=True)
print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,}")

Train: 51,558 | Val: 2,593


# سلول ۱۲ — آموزش + ذخیره checkpoint


In [12]:
from torch.amp import autocast, GradScaler

def label_smoothed_nll_loss(log_probs, target, smoothing=0.1, ignore_index=-100):
    N, V = log_probs.size()
    mask = (target != ignore_index).float()
    with torch.no_grad():
        true_dist = torch.zeros_like(log_probs)
        true_dist.fill_(smoothing / (V - 1))
        safe = target.clone(); safe[safe == ignore_index] = 0
        true_dist.scatter_(1, safe.unsqueeze(1), 1.0 - smoothing)
    loss = -(true_dist * log_probs).sum(dim=1)
    return (loss * mask).sum() / mask.sum().clamp(min=1)


model = Transformer(**model_cfg.to_dict()).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params:,}")
assert n_params > 2_000_000, f"Too few params ({n_params}) — deepcopy broken?"

optimizer = torch.optim.AdamW(model.parameters(), lr=train_cfg.lr,
                               betas=(0.9, 0.98), eps=1e-9, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: min(max(s, 1)**-0.5, max(s, 1) * (train_cfg.warmup_steps**-1.5))
)
scaler = GradScaler('cuda', enabled=(device.type == "cuda"))

CKPT_PATH = CKPT_DIR / "best_model.pt"
best_val, patience_counter = float("inf"), 0

print("=" * 65)
print(f"Training on {len(train_ds):,} samples | {train_cfg.epochs} epochs")
print("=" * 65)

for epoch in range(train_cfg.epochs):
    model.train()
    train_loss, n_batches, t0 = 0.0, 0, time.time()
    optimizer.zero_grad(set_to_none=True)
    for batch in train_loader:
        src      = batch["src"].to(device, non_blocking=True)
        tgt      = batch["tgt"].to(device, non_blocking=True)
        src_mask = batch["src_mask"].to(device, non_blocking=True)
        tgt_mask = batch["tgt_mask"].to(device, non_blocking=True)
        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
        with autocast('cuda', enabled=(device.type == 'cuda')):
            log_probs = model(src, tgt_in, src_mask, tgt_mask[:, :-1, :-1])
            loss = label_smoothed_nll_loss(log_probs.reshape(-1, vocab.size),
                                           tgt_out.reshape(-1),
                                           smoothing=train_cfg.label_smoothing,
                                           ignore_index=vocab.pad_idx)
        scaler.scale(loss / train_cfg.grad_accum_steps).backward()
        train_loss += loss.item(); n_batches += 1
        if n_batches % train_cfg.grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            optimizer.zero_grad(set_to_none=True)
    avg_train = train_loss / max(n_batches, 1)

    model.eval()
    val_loss, n_val = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            src      = batch["src"].to(device, non_blocking=True)
            tgt      = batch["tgt"].to(device, non_blocking=True)
            src_mask = batch["src_mask"].to(device, non_blocking=True)
            tgt_mask = batch["tgt_mask"].to(device, non_blocking=True)
            with autocast('cuda', enabled=(device.type == 'cuda')):
                log_probs = model(src, tgt[:, :-1], src_mask, tgt_mask[:, :-1, :-1])
                v_loss = label_smoothed_nll_loss(log_probs.reshape(-1, vocab.size),
                                                 tgt[:, 1:].reshape(-1),
                                                 smoothing=train_cfg.label_smoothing,
                                                 ignore_index=vocab.pad_idx)
            val_loss += v_loss.item(); n_val += 1
    avg_val = val_loss / max(n_val, 1)

    saved = ""
    if avg_val < best_val - 1e-4:
        best_val = avg_val; patience_counter = 0
        # ⭐ ذخیره با model_cfg.to_dict()
        torch.save({
            "model_cfg": model_cfg.to_dict(),
            "vocab_size": vocab.size,
            "model_state_dict": model.state_dict(),
            "best_val_loss": best_val,
            "epoch": epoch,
        }, CKPT_PATH)
        saved = " [saved]"
    else:
        patience_counter += 1
    print(f"Epoch {epoch+1:02d}/{train_cfg.epochs} | train={avg_train:.4f} | "
          f"val={avg_val:.4f} | {time.time()-t0:.1f}s{saved}")
    if patience_counter >= train_cfg.patience:
        print(f"Early stopping at epoch {epoch+1}"); break

print(f"\n✅ Best val loss: {best_val:.4f}")
print(f"✅ Checkpoint: {CKPT_PATH}")

Model params: 4,061,579
Training on 51,558 samples | 15 epochs
Epoch 01/15 | train=4.9031 | val=4.4143 | 31.8s [saved]
Epoch 02/15 | train=3.4976 | val=2.9117 | 31.0s [saved]
Epoch 03/15 | train=2.7641 | val=2.6350 | 30.7s [saved]
Epoch 04/15 | train=2.5871 | val=2.4912 | 30.8s [saved]
Epoch 05/15 | train=2.4653 | val=2.3628 | 30.8s [saved]
Epoch 06/15 | train=2.3619 | val=2.2553 | 30.9s [saved]
Epoch 07/15 | train=2.2787 | val=2.1655 | 30.7s [saved]
Epoch 08/15 | train=2.2070 | val=2.0864 | 30.7s [saved]
Epoch 09/15 | train=2.1455 | val=2.0182 | 30.8s [saved]
Epoch 10/15 | train=2.0923 | val=1.9564 | 30.9s [saved]
Epoch 11/15 | train=2.0467 | val=1.9027 | 30.9s [saved]
Epoch 12/15 | train=2.0051 | val=1.8582 | 30.6s [saved]
Epoch 13/15 | train=1.9667 | val=1.8131 | 30.7s [saved]
Epoch 14/15 | train=1.9312 | val=1.7693 | 31.2s [saved]
Epoch 15/15 | train=1.8983 | val=1.7314 | 30.9s [saved]

✅ Best val loss: 1.7314
✅ Checkpoint: /content/drive/MyDrive/persian_text_pipeline/checkpoints/b

# سلول ۱۳ — Load + Inference + Confidence Guard


In [13]:
from difflib import SequenceMatcher

_model = None
_model_meta = {}

def load_model(path=CKPT_PATH):
    global _model, _model_meta
    if _model is not None: return _model
    if not Path(path).exists():
        logger.warning(f"Checkpoint not found: {path}"); return None
    ckpt = torch.load(path, map_location=device)
    m = Transformer(**ckpt["model_cfg"]).to(device)
    m.load_state_dict(ckpt["model_state_dict"])
    m.eval()
    _model = m
    _model_meta = {"val_loss": ckpt.get("best_val_loss"), "cfg": ckpt["model_cfg"]}
    logger.info(f"Model loaded | val={ckpt.get('best_val_loss')}")
    return m


def _char_sim(a, b):
    if not a and not b: return 1.0
    if not a or not b: return 0.0
    return SequenceMatcher(None, a, b).ratio()


def _word_overlap(a, b):
    wa, wb = set(a.split()), set(b.split())
    return len(wa & wb) / len(wa) if wa else 1.0


def _greedy_decode(model, src_ids, max_len=64):
    src = torch.tensor([src_ids], dtype=torch.long, device=device)
    src_mask = torch.ones(1, 1, src.size(1), dtype=torch.bool, device=device)
    with torch.no_grad():
        memory = model.encode(src, src_mask)
        ys = torch.tensor([[vocab.sos_idx]], dtype=torch.long, device=device)
        for _ in range(max_len):
            tgt_mask = ~torch.triu(torch.ones(1, ys.size(1), ys.size(1),
                                              dtype=torch.bool, device=device), diagonal=1)
            dec_out = model.decode(memory, src_mask, ys, tgt_mask)
            next_id = int(model.generator(dec_out[:, -1]).argmax(-1).item())
            ys = torch.cat([ys, torch.tensor([[next_id]], device=device)], dim=1)
            if next_id == vocab.eos_idx: break
    return ys[0].tolist()[1:-1]


def correct(text, min_char_sim=0.85, min_word_overlap=0.7, min_len_ratio=0.7):
    """سه لایه گارد: char_sim، word_overlap، length_ratio."""
    t0 = time.time()
    model = load_model()
    if model is None:
        return {"text": text, "corrected": False, "reason": "model_not_loaded", "time_ms": 0}
    src_ids = vocab.encode(text[:64])
    if not src_ids:
        return {"text": text, "corrected": False, "reason": "empty_input", "time_ms": 0}
    try:
        with torch.no_grad():
            out_ids = _greedy_decode(model, src_ids)
        out_text = vocab.decode(out_ids) or text
        sim = _char_sim(text, out_text)
        if sim < min_char_sim:
            return {"text": text, "corrected": False, "reason": "low_char_sim",
                    "confidence": round(sim, 3), "candidate": out_text,
                    "time_ms": int((time.time()-t0)*1000)}
        ov = _word_overlap(text, out_text)
        if ov < min_word_overlap:
            return {"text": text, "corrected": False, "reason": "low_word_overlap",
                    "confidence": round(ov, 3), "candidate": out_text,
                    "time_ms": int((time.time()-t0)*1000)}
        ratio = len(out_text) / max(len(text), 1)
        if ratio < min_len_ratio or ratio > 1.5:
            return {"text": text, "corrected": False, "reason": "bad_length_ratio",
                    "confidence": round(ratio, 3), "candidate": out_text,
                    "time_ms": int((time.time()-t0)*1000)}
        return {"text": out_text, "corrected": out_text != text,
                "confidence": round(sim, 3), "time_ms": int((time.time()-t0)*1000)}
    except Exception as e:
        logger.error(f"Inference failed: {e}", exc_info=True)
        return {"text": text, "corrected": False, "reason": f"error: {e}", "time_ms": 0}


def pipeline(text, opts=None):
    t0 = time.time()
    san = sanitize(text, opts)
    corr = correct(san["text"])
    return {"original": text, "sanitized": san["text"], "corrected": corr["text"],
            "sanitizer_steps": san["steps_applied"],
            "corrected_flag": corr.get("corrected", False),
            "time_ms": int((time.time()-t0)*1000)}


print("✅ Inference ready")

✅ Inference ready


# سلول ۱۴ — FastAPI + Pydantic


In [14]:
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.middleware.gzip import GZipMiddleware
from pydantic import BaseModel, Field, field_validator


class SanitizeOptionsIn(BaseModel):
    apply_zwnj: bool = True
    apply_digits: bool = True
    strip_urls: bool = True
    strip_html: bool = True
    collapse_repeats: bool = True
    remove_diacritics: bool = True


class TextIn(BaseModel):
    text: str = Field(..., min_length=1, max_length=5000)
    options: SanitizeOptionsIn = Field(default_factory=SanitizeOptionsIn)
    @field_validator("text")
    @classmethod
    def _no_null(cls, v):
        if "\x00" in v: raise ValueError("Null byte not allowed")
        return v


class TextOut(BaseModel):
    text: str; changed: bool; steps_applied: list[str]; time_ms: int


class CorrectOut(BaseModel):
    text: str; corrected: bool; confidence: float | None = None
    reason: str | None = None; time_ms: int


class PipelineOut(BaseModel):
    original: str; sanitized: str; corrected: str
    sanitizer_steps: list[str]; corrected_flag: bool; time_ms: int


class HealthOut(BaseModel):
    status: str; device: str; model_loaded: bool; vocab_size: int


app = FastAPI(title="Persian Text Pipeline", version="1.0.0",
              description="Rule-based Persian text sanitizer + optional small spell corrector.")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
app.add_middleware(GZipMiddleware, minimum_size=500)


@app.middleware("http")
async def timing(request: Request, call_next):
    t0 = time.time()
    resp = await call_next(request)
    resp.headers["X-Process-Time-Ms"] = str(int((time.time()-t0)*1000))
    return resp


@app.get("/", tags=["meta"])
def root():
    return {"name": "persian-text-pipeline",
            "endpoints": ["/health", "/sanitize", "/correct", "/pipeline", "/docs"]}


@app.get("/health", response_model=HealthOut, tags=["meta"])
def health():
    return HealthOut(status="ok", device=device.type,
                     model_loaded=_model is not None, vocab_size=vocab.size)


@app.post("/sanitize", response_model=TextOut, tags=["sanitize"])
def api_sanitize(body: TextIn):
    t0 = time.time()
    r = sanitize(body.text, SanitizeOptions(**body.options.model_dump()))
    return TextOut(text=r["text"], changed=r["changed"],
                   steps_applied=r["steps_applied"], time_ms=int((time.time()-t0)*1000))


@app.post("/correct", response_model=CorrectOut, tags=["correct"])
async def api_correct(body: TextIn):
    r = await asyncio.to_thread(correct, body.text)
    return CorrectOut(**{k: r.get(k) for k in ["text","corrected","confidence","reason","time_ms"]})


@app.post("/pipeline", response_model=PipelineOut, tags=["pipeline"])
async def api_pipeline(body: TextIn):
    opts = SanitizeOptions(**body.options.model_dump())
    r = await asyncio.to_thread(pipeline, body.text, opts)
    return PipelineOut(**r)


print("✅ FastAPI app ready")

✅ FastAPI app ready


# سلول ۱۵ — اجرای سرور + cloudflared (بدون ngrok)


In [15]:
import nest_asyncio
nest_asyncio.apply()

subprocess.run(["pkill", "-f", "cloudflared"], check=False)

def _find_port(start=8000):
    for p in range(start, start + 20):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                s.bind(("0.0.0.0", p)); return p
            except OSError: continue
    raise RuntimeError("No free port")


PORT = _find_port()
print(f"Port: {PORT}")

def _serve():
    loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop)
    import uvicorn
    cfg = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning", loop="none")
    server = uvicorn.Server(cfg)
    server.install_signal_handlers = lambda: None
    loop.run_until_complete(server.serve())

threading.Thread(target=_serve, daemon=True).start()

import requests
for _ in range(150):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=1).status_code == 200: break
    except Exception: pass
    time.sleep(0.1)
else:
    raise RuntimeError("Uvicorn failed to start")
print("✅ Uvicorn is up")

CF = "/usr/local/bin/cloudflared"
if not Path(CF).exists():
    print("Downloading cloudflared...")
    subprocess.run(["wget","-q","-c",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", CF], check=False)
    os.chmod(CF, 0o755)

CF_LOG = "/tmp/cf.log"
open(CF_LOG, "w").close()
subprocess.Popen([CF, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                 stdout=open(CF_LOG, "w"), stderr=subprocess.STDOUT)

PUBLIC_URL = None
for _ in range(60):
    time.sleep(0.5)
    with open(CF_LOG) as f:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", f.read())
    if m: PUBLIC_URL = m.group(0); break

if PUBLIC_URL:
    globals()["PUBLIC_URL"] = PUBLIC_URL
    print("=" * 65)
    print(f"🌐 Public URL : {PUBLIC_URL}")
    print(f"📖 Docs       : {PUBLIC_URL}/docs")
    print(f"❤️  Health     : {PUBLIC_URL}/health")
    print("=" * 65)
else:
    print(f"⚠️ No tunnel. Local: http://127.0.0.1:{PORT}")

Port: 8000
✅ Uvicorn is up
🌐 Public URL : https://assume-shell-fisheries-jacob.trycloudflare.com
📖 Docs       : https://assume-shell-fisheries-jacob.trycloudflare.com/docs
❤️  Health     : https://assume-shell-fisheries-jacob.trycloudflare.com/health


# سلول ۱۶ — تست کامل


In [16]:
import requests, json
BASE = PUBLIC_URL

def call(path, payload=None, method="POST"):
    try:
        if method == "POST":
            r = requests.post(f"{BASE}{path}", json=payload, timeout=30)
        else:
            r = requests.get(f"{BASE}{path}", timeout=15)
        return r.status_code, r.json() if "json" in r.headers.get("content-type","") else {"_raw": r.text[:200]}
    except Exception as e:
        return None, {"error": str(e)}


print("=" * 65)
print("  1) HEALTH (before)")
print("=" * 65)
print(call("/health", method="GET"))

print("\n" + "=" * 65)
print("  2) SANITIZE")
print("=" * 65)
print(call("/sanitize", {"text": "سلام   می روم كتَب هاي جديد http://x.com <b>خوب</b> ٣ عدد ۱۲۳۴"}))

print("\n" + "=" * 65)
print("  3) CORRECT — in-distribution")
print("=" * 65)
print(call("/correct", {"text": "برف روی کوه‌ها نشستة است"}))

print("\n" + "=" * 65)
print("  4) CORRECT — OOD (should reject gracefully)")
print("=" * 65)
print(call("/correct", {"text": "من به مدرصه می‌روم"}))

print("\n" + "=" * 65)
print("  5) HEALTH (after)")
print("=" * 65)
print(call("/health", method="GET"))

print("\n" + "=" * 65)
print("  6) PIPELINE")
print("=" * 65)
print(call("/pipeline", {"text": "می روم به مدرصه"}))

print("\n" + "=" * 65)
print(f"  ✅ Done")
print(f"  📖 Docs  : {BASE}/docs")
print(f"  ❤️  Health: {BASE}/health")
print("=" * 65)

  1) HEALTH (before)
(200, {'status': 'ok', 'device': 'cuda', 'model_loaded': False, 'vocab_size': 139})

  2) SANITIZE
(200, {'text': 'سلام می\u200cروم کتب\u200cهای جدید خوب 3 عدد 1234', 'changed': True, 'steps_applied': ['html', 'urls_emails', 'ar2fa', 'diacritics', 'digits', 'zwnj', 'repeats', 'whitespace'], 'time_ms': 0})

  3) CORRECT — in-distribution
(200, {'text': 'برف روی کوه\u200cها نشستة است', 'corrected': False, 'confidence': 0.588, 'reason': 'low_char_sim', 'time_ms': 264})

  4) CORRECT — OOD (should reject gracefully)
(200, {'text': 'من به مدرصه می\u200cروم', 'corrected': False, 'confidence': 0.703, 'reason': 'low_char_sim', 'time_ms': 92})

  5) HEALTH (after)
(200, {'status': 'ok', 'device': 'cuda', 'model_loaded': True, 'vocab_size': 139})

  6) PIPELINE
(200, {'original': 'می روم به مدرصه', 'sanitized': 'می\u200cروم به مدرصه', 'corrected': 'می\u200cروم به مدرصه', 'sanitizer_steps': ['zwnj'], 'corrected_flag': False, 'time_ms': 112})

  ✅ Done
  📖 Docs  : https://assu